# 🔬 Model Analysis — Global Fuel Price Predictor

Notebook ini melakukan **analisis error mendalam** terhadap tiga model regresi
(**KNN**, **SVM**, **Random Forest**) yang telah dilatih oleh `train.py`.

Alur analisis:
1. Muat model terlatih & test set (`data/test_data.pkl`).
2. Bandingkan prediksi ketiga model (scatter 3-panel).
3. Error per **region** (heatmap RMSE).
4. Error per **income_level** (bar chart MAE).
5. Error per **subsidy_level**.
6. Distribusi residual model terbaik (histogram + Q-Q plot).
7. Actual vs predicted model terbaik, diwarnai per region.
8. 5 prediksi terburuk beserta fitur input-nya.
9. Kesimpulan & rekomendasi kuantitatif.

## 0. Setup

Import library, konfigurasi path agar package `src` dapat diimpor dari folder
`notebooks/`, serta penyetelan style plot sesuai *design system* project
(`#17171c`, `#003c33`, `#ff7759`, `#1863dc`).

In [ ]:
import os
import sys
import json

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from scipy import stats

# Make the project root (parent of notebooks/) importable.
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.basename(os.getcwd()) != "notebooks":
    # Allow running from the project root too.
    ROOT = os.path.abspath(os.getcwd())
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.models import KNNModel, SVMModel, RandomForestModel
from src.models._common import compute_metrics

# Design-system palette.
PRIMARY, DEEP_GREEN, CORAL, ACTION_BLUE = "#17171c", "#003c33", "#ff7759", "#1863dc"
MUTED = "#93939f"
MODEL_COLOURS = {"KNN": DEEP_GREEN, "SVM": CORAL, "Random Forest": ACTION_BLUE}

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")
plt.rcParams["figure.dpi"] = 110
print("Setup complete. Project root:", ROOT)

## 1. Muat model terlatih & test data

Test set disimpan oleh `DataPreprocessor.fit_transform()` ke
`data/test_data.pkl` dan berisi `X_test`, `y_test`, `feature_names`, serta
`df_test` (baris mentah dengan kolom `region`, `income_level`, `subsidy_level`).
Ini memastikan analisis dilakukan pada **test set yang sama** untuk ketiga model.

In [ ]:
DATA_DIR = os.path.join(ROOT, "data")
MODELS_DIR = os.path.join(ROOT, "models")

bundle = joblib.load(os.path.join(DATA_DIR, "test_data.pkl"))
X_test = bundle["X_test"]
y_test = np.asarray(bundle["y_test"], dtype=float)
feature_names = bundle["feature_names"]
df_test = bundle["df_test"].reset_index(drop=True)

models = {
    "KNN": KNNModel.load(os.path.join(MODELS_DIR, "knn_model.pkl")),
    "SVM": SVMModel.load(os.path.join(MODELS_DIR, "svm_model.pkl")),
    "Random Forest": RandomForestModel.load(os.path.join(MODELS_DIR, "rf_model.pkl")),
}

# Predictions on the shared test set.
preds = {name: m.predict(X_test) for name, m in models.items()}

with open(os.path.join(DATA_DIR, "model_comparison.json"), "r", encoding="utf-8") as fh:
    comparison = json.load(fh)

print("Test set:", X_test.shape, "| features:", len(feature_names))
print("Models loaded:", list(models.keys()))
metrics_table = pd.DataFrame({n: compute_metrics(y_test, p) for n, p in preds.items()}).T
metrics_table.round(4)

## 2. Perbandingan prediksi ketiga model (scatter 3-panel)

Setiap panel memplot **actual vs predicted** untuk satu model. Semakin rapat
titik mengikuti garis ideal `y = x` (garis coral), semakin akurat model tersebut.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.6), sharex=True, sharey=True)
lo = float(min(y_test.min(), min(p.min() for p in preds.values())))
hi = float(max(y_test.max(), max(p.max() for p in preds.values())))

for ax, (name, p) in zip(axes, preds.items()):
    ax.scatter(y_test, p, s=10, alpha=0.35, color=MODEL_COLOURS[name], edgecolors="none")
    ax.plot([lo, hi], [lo, hi], ls="--", lw=2, color=CORAL)
    r2 = compute_metrics(y_test, p)["R2"]
    ax.set_title(f"{name}  (R² = {r2:.3f})", color=PRIMARY, fontweight="bold")
    ax.set_xlabel("Actual (USD/L)", color=PRIMARY)
axes[0].set_ylabel("Predicted (USD/L)", color=PRIMARY)
fig.suptitle("Actual vs Predicted — perbandingan model", fontsize=15,
             fontweight="bold", color=PRIMARY)
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## 3. Error analysis per **region** — heatmap RMSE

RMSE dihitung per (region × model). Heatmap memperlihatkan region mana yang
paling sulit diprediksi dan apakah ada model yang konsisten lebih baik di
seluruh region.

In [ ]:
def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

regions = sorted(df_test["region"].unique())
rmse_region = pd.DataFrame(index=regions, columns=list(models.keys()), dtype=float)
for r in regions:
    mask = (df_test["region"] == r).to_numpy()
    for name, p in preds.items():
        rmse_region.loc[r, name] = rmse(y_test[mask], p[mask])

fig, ax = plt.subplots(figsize=(8, 6))
data = rmse_region.astype(float).to_numpy()
im = ax.imshow(data, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(len(models))); ax.set_xticklabels(list(models.keys()))
ax.set_yticks(range(len(regions))); ax.set_yticklabels(regions)
for i in range(len(regions)):
    for j in range(len(models)):
        ax.text(j, i, f"{data[i, j]:.3f}", ha="center", va="center",
                color="black", fontsize=9)
ax.set_title("RMSE per Region per Model", color=PRIMARY, fontweight="bold")
fig.colorbar(im, ax=ax, label="RMSE (USD/L)")
fig.tight_layout()
plt.show()
rmse_region.astype(float).round(4)

## 4. Error analysis per **income_level** — bar chart MAE

MAE dikelompokkan berdasarkan tingkat pendapatan negara (`Low`, `Middle`,
`High`). Membantu melihat apakah model bias pada kelompok pendapatan tertentu.

In [ ]:
def mae(a, b):
    return float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))

order_income = ["Low", "Middle", "High"]
levels = [l for l in order_income if l in df_test["income_level"].unique()]
mae_income = pd.DataFrame(index=levels, columns=list(models.keys()), dtype=float)
for lvl in levels:
    mask = (df_test["income_level"] == lvl).to_numpy()
    for name, p in preds.items():
        mae_income.loc[lvl, name] = mae(y_test[mask], p[mask])

ax = mae_income.plot(kind="bar", figsize=(9, 5.5),
                     color=[MODEL_COLOURS[m] for m in models])
ax.set_title("MAE per Income Level", color=PRIMARY, fontweight="bold")
ax.set_ylabel("MAE (USD/L)", color=PRIMARY)
ax.set_xlabel("Income level", color=PRIMARY)
ax.legend(title="Model", frameon=False)
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()
mae_income.round(4)

## 5. Error analysis per **subsidy_level**

Analisis serupa untuk tingkat subsidi (`Low`, `Medium`, `High`, `Very High`) —
negara dengan subsidi tinggi cenderung memiliki harga sangat rendah, sehingga
menarik untuk dicek apakah error meningkat di kelompok ini.

In [ ]:
order_subsidy = ["Low", "Medium", "High", "Very High"]
slevels = [l for l in order_subsidy if l in df_test["subsidy_level"].unique()]
mae_subsidy = pd.DataFrame(index=slevels, columns=list(models.keys()), dtype=float)
for lvl in slevels:
    mask = (df_test["subsidy_level"] == lvl).to_numpy()
    for name, p in preds.items():
        mae_subsidy.loc[lvl, name] = mae(y_test[mask], p[mask])

ax = mae_subsidy.plot(kind="bar", figsize=(9.5, 5.5),
                      color=[MODEL_COLOURS[m] for m in models])
ax.set_title("MAE per Subsidy Level", color=PRIMARY, fontweight="bold")
ax.set_ylabel("MAE (USD/L)", color=PRIMARY)
ax.set_xlabel("Subsidy level", color=PRIMARY)
ax.legend(title="Model", frameon=False)
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()
mae_subsidy.round(4)

## 6. Distribusi residual — model terbaik (histogram + Q-Q plot)

Model terbaik dipilih berdasarkan **RMSE terendah** pada test set. Histogram
residual idealnya simetris di sekitar 0; Q-Q plot menilai seberapa dekat
residual terhadap distribusi normal (titik mengikuti garis = mendekati normal).

In [ ]:
test_rmse = {name: rmse(y_test, p) for name, p in preds.items()}
best_model = min(test_rmse, key=test_rmse.get)
print("Model terbaik (RMSE terendah):", best_model, "-> RMSE =", round(test_rmse[best_model], 5))

residuals = y_test - preds[best_model]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
axes[0].hist(residuals, bins=45, color=DEEP_GREEN, alpha=0.85, edgecolor="white")
axes[0].axvline(0, color=CORAL, lw=2, ls="--")
axes[0].set_title(f"Residual Distribution — {best_model}", color=PRIMARY, fontweight="bold")
axes[0].set_xlabel("Residual (Actual − Predicted)", color=PRIMARY)
axes[0].set_ylabel("Frequency", color=PRIMARY)

stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].get_lines()[0].set_markerfacecolor(ACTION_BLUE)
axes[1].get_lines()[0].set_markeredgecolor("none")
axes[1].get_lines()[0].set_markersize(4)
axes[1].get_lines()[1].set_color(CORAL)
axes[1].set_title(f"Q-Q Plot — {best_model}", color=PRIMARY, fontweight="bold")
fig.tight_layout(); plt.show()

## 7. Actual vs Predicted (model terbaik) — diwarnai per region

Memvisualkan bagaimana akurasi model terbaik bervariasi antar region. Cluster
yang menyimpang jauh dari garis ideal menandakan region yang lebih sulit.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
palette = [DEEP_GREEN, CORAL, ACTION_BLUE, PRIMARY, "#6a994e", "#9b5de5", "#f4a261"]
for color, r in zip(palette, sorted(df_test["region"].unique())):
    mask = (df_test["region"] == r).to_numpy()
    ax.scatter(y_test[mask], preds[best_model][mask], s=14, alpha=0.5,
               color=color, edgecolors="none", label=r)
lo = float(min(y_test.min(), preds[best_model].min()))
hi = float(max(y_test.max(), preds[best_model].max()))
ax.plot([lo, hi], [lo, hi], ls="--", lw=2, color="#000000")
ax.set_title(f"Actual vs Predicted by Region — {best_model}", color=PRIMARY, fontweight="bold")
ax.set_xlabel("Actual (USD/L)", color=PRIMARY)
ax.set_ylabel("Predicted (USD/L)", color=PRIMARY)
ax.legend(title="Region", frameon=False, fontsize=9)
fig.tight_layout(); plt.show()

## 8. Lima prediksi terburuk (error absolut terbesar)

Menampilkan baris dengan |actual − predicted| terbesar untuk model terbaik,
lengkap dengan fitur input mentahnya — berguna untuk mendiagnosis *kapan* model
gagal.

In [ ]:
errors = np.abs(y_test - preds[best_model])
worst_idx = np.argsort(errors)[::-1][:5]
worst = df_test.iloc[worst_idx][[
    "date", "country", "region", "income_level", "subsidy_level",
    "brent_crude_usd", "tax_percentage"]].copy()
worst["actual"] = np.round(y_test[worst_idx], 3)
worst["predicted"] = np.round(preds[best_model][worst_idx], 3)
worst["abs_error"] = np.round(errors[worst_idx], 3)
worst.reset_index(drop=True)

## 9. Kesimpulan & rekomendasi kuantitatif

Ringkasan metrik test set ketiga model dan rekomendasi berbasis angka.

In [ ]:
summary = pd.DataFrame({n: compute_metrics(y_test, p) for n, p in preds.items()}).T
summary = summary.round(4).sort_values("RMSE")
print("Ranking model (RMSE menaik):")
display(summary)

best = summary.index[0]
b = summary.loc[best]
print(f"\nREKOMENDASI: {best}")
print(f"  - RMSE  = {b['RMSE']:.4f} USD/L (terendah)")
print(f"  - MAE   = {b['MAE']:.4f} USD/L")
print(f"  - R²    = {b['R2']:.4f}")
print(f"  - MAPE  = {b['MAPE']:.2f}%")

### Interpretasi

- **Model terpilih** adalah yang memiliki **RMSE terendah** pada test set (lihat
  output di atas) — konsisten dengan `best_model` pada `data/model_comparison.json`.
- **Random Forest** umumnya unggul pada data tabular karena mampu menangkap
  interaksi non-linier antar fitur (mis. `country` × `subsidy_level` × `brent`)
  tanpa memerlukan penskalaan, sekaligus menyediakan *feature importance*.
- **KNN** dapat sangat kompetitif di sini karena `country` (label-encoded)
  praktis mengidentifikasi level harga tiap negara, sehingga tetangga terdekat
  sangat informatif — namun sensitif terhadap skala & lambat saat inferensi.
- **SVM (SVR)** dilatih pada subsample 10.000 baris; bila RMSE-nya tertinggal,
  penyebab utamanya adalah penskalaan fitur kategorikal (`country` 0–83 tidak
  ternormalisasi) yang mendominasi kernel jarak.

> **Catatan reproducibility:** seluruh `random_state = 42`, split 80:20,
> dan test set identik untuk ketiga model.